# 03 — Resource-Constrained Experiment
Dropped features: BS, Gestational Diabetes, Preexisting Diabetes, Mental Health
Pipeline: StandardScaler → PCA(0.95) → SMOTE → All Models

In [5]:
!git clone https://github.com/alifnw/3-Alif-Livyandra.git
%cd 3-Alif-Livyandra

Cloning into '3-Alif-Livyandra'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 60 (delta 21), reused 25 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 6.05 MiB | 10.15 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/3-Alif-Livyandra


In [1]:
import warnings; warnings.filterwarnings("ignore")
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline        import Pipeline as SkPipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.decomposition   import PCA
from sklearn.linear_model    import LogisticRegression, SGDClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import (RandomForestClassifier, GradientBoostingClassifier,
                                     AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier)
from sklearn.svm             import SVC
from sklearn.metrics         import (make_scorer, accuracy_score, precision_score,
                                     recall_score, f1_score, classification_report)
from xgboost                 import XGBClassifier
from lightgbm                import LGBMClassifier
from imblearn.pipeline       import Pipeline as ImbPipeline
from imblearn.over_sampling  import SMOTE

RS       = 42
N_SPLITS = 10
SKF      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RS)

SCORING = {
    "accuracy"   : make_scorer(accuracy_score),
    "precision_1": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_1"   : make_scorer(recall_score,    pos_label=1, zero_division=0),
    "f1_1"       : make_scorer(f1_score,        pos_label=1, zero_division=0),
}

In [6]:
df = pd.read_csv('/content/3-Alif-Livyandra/data/processed/maternal_health_cleaned.csv')
print(df.shape)

(1168, 12)


## Feature Engineering

In [ ]:
df["Is_Fever"] = (df["Body Temp"] > 99).astype(int)

binary_cols  = [col for col in df.select_dtypes(include=[np.number]).columns
                if df[col].dropna().isin([0, 1]).all()]
numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                if col not in binary_cols]

feature_cols = numeric_cols + binary_cols
X = df[feature_cols]
y = df["Risk Level"].map({"High": 1, "Low": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RS, stratify=y
)
print(f"train: {X_train.shape[0]} | test: {X_test.shape[0]}")

## Feature Engineering Variants (RC versions)

In [ ]:
RC_DROP_COLS = ["BS", "Gestational Diabetes", "Preexisting Diabetes", "Mental Health"]

# fe-a rc: all features minus dropped columns
X_train_A_rc = X_train.drop(columns=[c for c in RC_DROP_COLS if c in X_train.columns]).copy()
X_test_A_rc  = X_test.drop(columns=[c for c in RC_DROP_COLS if c in X_test.columns]).copy()

# fe-b rc: also drop body temp
X_train_B_rc = X_train_A_rc.drop(columns=["Body Temp"], errors="ignore").copy()
X_test_B_rc  = X_test_A_rc.drop(columns=["Body Temp"],  errors="ignore").copy()

FE_RC = {"FE-A (RC)": (X_train_A_rc, X_test_A_rc), "FE-B (RC)": (X_train_B_rc, X_test_B_rc)}

for name, (tr, te) in FE_RC.items():
    print(f"{name}: train {tr.shape}  features: {tr.columns.tolist()}")

## Model Registry

In [ ]:
BASE_MODELS = {
    "Logistic Regression": LogisticRegression(random_state=RS, max_iter=1000),
    "Decision Tree"      : DecisionTreeClassifier(random_state=RS),
    "Random Forest"      : RandomForestClassifier(random_state=RS, n_jobs=-1),
    "Gradient Boosting"  : GradientBoostingClassifier(random_state=RS),
    "XGBoost"            : XGBClassifier(random_state=RS, verbosity=0, eval_metric="logloss", n_jobs=-1),
    "LightGBM"           : LGBMClassifier(random_state=RS, verbose=-1, n_jobs=-1),
    "AdaBoost"           : AdaBoostClassifier(random_state=RS),
    "SVM"                : SVC(random_state=RS, probability=True),
    "Bagging"            : BaggingClassifier(random_state=RS, n_jobs=-1),
    "Extra Trees"        : ExtraTreesClassifier(random_state=RS, n_jobs=-1),
    "SGD"                : SGDClassifier(random_state=RS, loss="log_loss", max_iter=1000),
}

## RC Pipeline Builder

In [ ]:
def build_rc_pipeline(model):
    # pipeline: scaler -> pca(0.95) -> smote -> classifier
    # pca removes categorical structure so smote-nc is not applicable
    clf = copy.deepcopy(model)
    smote = SMOTE(random_state=RS)
    steps = [
        ("scaler",   StandardScaler()),
        ("pca",      PCA(n_components=0.95, random_state=RS)),
        ("resampler",smote),
        ("clf",      clf),
    ]
    return ImbPipeline(steps)

## Resource-Constrained Leaderboard

In [ ]:
rc_records = []
total = len(FE_RC) * len(BASE_MODELS)
done  = 0

for fe_name, (X_tr, X_te) in FE_RC.items():
    # fit a temp pipeline to report pca components
    temp_scaler = StandardScaler()
    temp_pca    = PCA(n_components=0.95, random_state=RS)
    temp_pca.fit(temp_scaler.fit_transform(X_tr))
    print(f"\n{fe_name}: {X_tr.shape[1]} features -> {temp_pca.n_components_} PCA components")

    for mdl_name, mdl in BASE_MODELS.items():
        done += 1
        try:
            pipe   = build_rc_pipeline(mdl)
            cv_res = cross_validate(pipe, X_tr, y_train, cv=SKF,
                                    scoring=SCORING, n_jobs=-1)
            row = {
                "FE"         : fe_name,
                "Model"      : mdl_name,
                "Accuracy"   : round(cv_res["test_accuracy"].mean(),    4),
                "Precision_1": round(cv_res["test_precision_1"].mean(), 4),
                "Recall_1"   : round(cv_res["test_recall_1"].mean(),    4),
                "F1_1"       : round(cv_res["test_f1_1"].mean(),        4),
            }
            rc_records.append(row)
            print(f"  [{done:>2}/{total}] {fe_name} | {mdl_name:<22} -> Recall_1={row['Recall_1']:.4f}")
        except Exception as exc:
            print(f"  SKIPPED {fe_name} | {mdl_name}: {exc}")

rc_leaderboard = pd.DataFrame(rc_records).sort_values("Recall_1", ascending=False).reset_index(drop=True)
print("\n--- Resource-Constrained Leaderboard ---")
print("Top 10:")
display(rc_leaderboard.head(10))
print("\nBottom 10:")
display(rc_leaderboard.tail(10))

## Test Set Evaluation: Best RC Model

In [ ]:
best_rc_row = rc_leaderboard.iloc[0]
best_rc_fe  = best_rc_row["FE"]
best_rc_mdl = best_rc_row["Model"]

X_tr_rc_best, X_te_rc_best = FE_RC[best_rc_fe]

final_rc_pipe = build_rc_pipeline(BASE_MODELS[best_rc_mdl])
final_rc_pipe.fit(X_tr_rc_best, y_train)
y_pred_rc = final_rc_pipe.predict(X_te_rc_best)

print(f"best RC model: {best_rc_mdl}  |  FE: {best_rc_fe}")
print(classification_report(y_test, y_pred_rc, target_names=["Low Risk", "High Risk"]))